# Contexte
 Ceci est notre notebook de prise en main afin de créer un RAG avec un corpus documentaire en utilisant Langchain. Notre pipeline RAG est une pipeline experimentale afin de prendre en main les différents outils.

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


-----------------------------------------------------------------

### Les composants clés de LangChain pour le RAG
Pour construire un système RAG avec LangChain, nous avons besoins des composants suivants :

- `Document Loaders` : Chargent les documents à partir de différentes sources (PDF, sites web, bases de données, etc.).
- `Text Splitters` : Divisent les documents en morceaux (chunks) plus petits pour une manipulation plus efficace.
- `Embeddings` : Convertissent le texte en vecteurs numériques pour permettre la recherche sémantique.
- `Vector Stores` : Stockent les embeddings pour une récupération rapide.
- `Retrievers` : Récupèrent les documents pertinents en fonction d'une requête.
- `LLMs (Language Models)` : Génèrent des réponses basées sur les documents récupérés.
- `Chains` : Orchestrent le flux d'informations entre les composants.


# RAG Langchain

### Prérequis

In [ ]:
%pip install pyPDF2
%pip install faiss-cpu
%pip install sentence_transformers
%pip install huggingface_hub
%pip install langchain-community
%pip install sentence-transformers
%apt-get install -y poppler-utils
%pip install pdf2image pillow matplotlib

### Étape 1 : Importation des bibliothèques nécessaires python

In [ ]:
from PyPDF2 import PdfReader
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.chains.question_answering import load_qa_chain
from langchain import HuggingFaceHub, PromptTemplate, LLMChain
from langchain.embeddings import HuggingFaceEmbeddings
from pdf2image import convert_from_path
import matplotlib.pyplot as plt
import os

In [ ]:
MYHFKEY = "MYHFKEY_HUGGINGFACEHUB_API_TOKEN"  # A remplacer par votre clé API Hugging Face
os.environ["HUGGINGFACEHUB_API_TOKEN"] = MYHFKEY

### Étape 2 : Lecture du PDF et extraction du texte

In [ ]:
# Lecture du PDF et extraction du texte
ata_folder = "data/"
allContent = []
metadata = []

# Lecture de tous les fichiers PDF dans le dossier "data"
for file_name in os.listdir(data_folder):
    if file_name.endswith(".pdf"):
        pdf_path = os.path.join(data_folder, file_name)
        try:
            reader = PdfReader(pdf_path)
            for i, page in enumerate(reader.pages):
                text = page.extract_text()
                if text.strip():  # Si le texte n'est pas vide
                    allContent.append(text)
                    metadata.append({"file_name": file_name, "page_number": i + 1, "pdf_path": pdf_path})
        except Exception as e:
            print(f"Erreur lors de la lecture du fichier {file_name}: {e}")

# Vérification du contenu
if not allContent:
    raise ValueError("Aucun contenu valide trouvé dans les fichiers PDF du dossier.")

### Étape 3 : Division du texte en morceaux (chunks)
Pour faciliter la gestion du texte, nous le divisons en morceaux plus petits.

In [ ]:
# Division du texte en morceaux (chunks)
finalAllContent = []
chunk_metadata = []

text_splitter = CharacterTextSplitter(separator="\n",
                                      chunk_size=200,
                                      chunk_overlap=20,
                                      length_function=len)

for i, text in enumerate(allContent):
    chunks = text_splitter.split_text(text)
    finalAllContent.extend(chunks)
    chunk_metadata.extend([metadata[i]] * len(chunks))  # Associer les métadonnées aux chunks

### Étape 4 : Création des embeddings
Nous utilisons un modèle d'embeddings multilingue adapté au français (de part le contexte du projet).

In [ ]:
# Création des embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

### Étape 5 : Création du vecteur store avec FAISS
Nous stockons les embeddings dans un index FAISS pour permettre une recherche rapide.

In [14]:
# Création du vecteur store avec FAISS
docSearch = FAISS.from_texts(finalAllContent, embeddings)

### Étape 6 : Configuration du modèle de langage (LLM)
Nous choisissons un modèle capable de comprendre et de générer du texte en français.

In [15]:
# Configuration du modèle de langage (LLM)
llm = HuggingFaceHub(repo_id="google/flan-t5-large",
                     model_kwargs={"temperature":1e-10})


<ipython-input-15-fb22f915c26e>:2: LangChainDeprecationWarning: The class `HuggingFaceHub` was deprecated in LangChain 0.0.21 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEndpoint``.
  llm = HuggingFaceHub(repo_id="google/flan-t5-large",


Nous n'avons pas pu utiliser des modèles mistral de part le taille des modèles et les puissances a notre disposition pour cette première prise en main.

### Étape 7 : Chargement de la chaîne de question-réponse

In [ ]:
# Chargement de la chaîne de question-réponse
newChain = load_qa_chain(llm, chain_type="stuff")

# Liste des questions à poser
questions = [
    "Quel a était la croissance des effectifs en 2023?",
    "Quel est le chiffre d'affaires d'EDF Renouvelables en 2023?"
]

# Boucle pour poser les questions
for question in questions:
    docs = docSearch.similarity_search(question)
    answer = newChain.run(input_documents=docs, question=question)
    print(f"Question: {question}")
    print(f"Réponse: {answer}\n")
